In [16]:
import requests
import pandas as pd
import time
import os
import json
from sqlalchemy import create_engine, inspect
from dotenv import load_dotenv

# ==========================================
# 🔑 ENV
# ==========================================
load_dotenv()

DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")
DB_HOST = os.getenv("DB_HOST")
DB_PORT = os.getenv("DB_PORT")
DB_NAME = os.getenv("DB_NAME")
API_KEY = os.getenv("API_KEY")

# ==========================================
# 🔌 CONEXÃO
# ==========================================
engine = create_engine(
    f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

inspector = inspect(engine)

HEADERS = {
    "x-rapidapi-key": API_KEY,
    "x-rapidapi-host": "v3.football.api-sports.io"
}

TEAM_ID = 131
LEAGUE_ID = 71

# ==========================================
# 🛠️ FUNÇÕES AUXILIARES
# ==========================================

def tabela_tem_dados(nome_tabela):
    try:
        df = pd.read_sql(f"SELECT 1 FROM {nome_tabela} LIMIT 1", engine)
        return not df.empty
    except:
        return False


def salvar_tabela(df, nome_tabela):
    if df is None or df.empty:
        print(f"⚠️ {nome_tabela} vazio")
        return

    print(f"💾 Salvando {nome_tabela} ({len(df)} linhas)")

    df.to_sql(
        nome_tabela,
        engine,
        if_exists="append",
        index=False
    )


def safe_request(url, params):
    try:
        res = requests.get(url, headers=HEADERS, params=params)
        data = res.json()

        if "errors" in data and data["errors"]:
            print("❌ Erro API:", data["errors"])
            return None

        return data.get("response", None)

    except Exception as e:
        print("❌ Erro request:", e)
        return None


# ==========================================
# 📅 FIXTURES
# ==========================================
fixtures_ids = {}

print("\n🔵 FIXTURES")

if tabela_tem_dados("fixtures"):
    print("⏭️ fixtures já existe, pulando...")

    # 🔥 carregar ids do banco
    try:
        df = pd.read_sql("SELECT fixture_id, season FROM fixtures", engine)

        for ano in df["season"].unique():
            fixtures_ids[ano] = df[df["season"] == ano]["fixture_id"].tolist()

    except Exception as e:
        print("❌ erro ao carregar fixtures:", e)

else:
    for ano in range(2022, 2025):
        res = safe_request(
            "https://v3.football.api-sports.io/fixtures",
            {"team": TEAM_ID, "season": ano}
        )

        if not res:
            continue

        df = pd.json_normalize(res)
        df["season"] = ano

        salvar_tabela(df, "fixtures")

        fixtures_ids[ano] = [j["fixture"]["id"] for j in res]

        time.sleep(1.2)


# ==========================================
# 🔁 LOOP BASE
# ==========================================
def loop_por_jogo(endpoint, nome_tabela, process_func):

    print(f"\n🔵 {nome_tabela}")

    if tabela_tem_dados(nome_tabela):
        print(f"⏭️ {nome_tabela} já existe, pulando...")
        return

    tabelas = []

    for ano, jogos in fixtures_ids.items():
        for jogo_id in jogos[:15]:

            res = safe_request(
                f"https://v3.football.api-sports.io{endpoint}",
                {"fixture": jogo_id}
            )

            if not res:
                continue

            df = process_func(res, jogo_id, ano)

            if df is not None and not df.empty:
                tabelas.append(df)

            time.sleep(1.2)

    if tabelas:
        salvar_tabela(pd.concat(tabelas, ignore_index=True), nome_tabela)


# ==========================================
# LINEUPS
# ==========================================
def process_lineups(res, jogo_id, ano):
    tabelas = []

    for team in res:
        df = pd.json_normalize(team["startXI"])
        df["team_id"] = team["team"]["id"]
        df["fixture_id"] = jogo_id
        df["season"] = ano
        tabelas.append(df)

    return pd.concat(tabelas) if tabelas else None

loop_por_jogo("/fixtures/lineups", "lineups", process_lineups)


# ==========================================
# EVENTS
# ==========================================
def process_events(res, jogo_id, ano):
    df = pd.json_normalize(res)
    df["fixture_id"] = jogo_id
    df["season"] = ano
    return df

loop_por_jogo("/fixtures/events", "events", process_events)


# ==========================================
# STATISTICS
# ==========================================
def process_stats(res, jogo_id, ano):
    tabelas = []

    for team in res:
        df = pd.json_normalize(team["statistics"])
        df["team_id"] = team["team"]["id"]
        df["fixture_id"] = jogo_id
        df["season"] = ano
        tabelas.append(df)

    return pd.concat(tabelas) if tabelas else None

loop_por_jogo("/fixtures/statistics", "statistics", process_stats)


# ==========================================
# PLAYERS
# ==========================================
def process_players(res, jogo_id, ano):
    tabelas = []

    for team in res:
        for player in team["players"]:
            df = pd.json_normalize(player)

            if "statistics" in df.columns:
                df["statistics"] = df["statistics"].apply(
                    lambda x: json.dumps(x) if x else None
                )

            df["team_id"] = team["team"]["id"]
            df["fixture_id"] = jogo_id
            df["season"] = ano

            tabelas.append(df)

    return pd.concat(tabelas) if tabelas else None

loop_por_jogo("/fixtures/players", "players", process_players)


# ==========================================
# STANDINGS
# ==========================================
print("\n🔵 standings")

if tabela_tem_dados("standings"):
    print("⏭️ standings já existe, pulando...")
else:
    tabelas = []

    for ano in range(2022, 2025):
        res = safe_request(
            "https://v3.football.api-sports.io/standings",
            {
                "league": LEAGUE_ID,
                "season": ano
            }
        )

        if not res:
            continue

        for item in res:
            tabela = item["league"]["standings"][0]
            df = pd.json_normalize(tabela)
            df["season"] = ano
            tabelas.append(df)

        time.sleep(1.2)

    if tabelas:
        salvar_tabela(pd.concat(tabelas), "standings")


# ==========================================
# TEAM STATISTICS
# ==========================================
print("\n🔵 team_statistics")

if tabela_tem_dados("team_statistics"):
    print("⏭️ team_statistics já existe, pulando...")
else:
    tabelas = []

    for ano in range(2022, 2025):
        res = safe_request(
            "https://v3.football.api-sports.io/teams/statistics",
            {
                "team": TEAM_ID,
                "league": LEAGUE_ID,
                "season": ano
            }
        )

        if not res:
            continue

        df = pd.json_normalize(res)
        df["season"] = ano
        tabelas.append(df)

        time.sleep(1.2)

    if tabelas:
        salvar_tabela(pd.concat(tabelas), "team_statistics")


🔵 FIXTURES
❌ Erro API: {'requests': 'You have reached the request limit for the day, Go to https://dashboard.api-football.com to upgrade your plan.'}
❌ Erro API: {'requests': 'You have reached the request limit for the day, Go to https://dashboard.api-football.com to upgrade your plan.'}
❌ Erro API: {'requests': 'You have reached the request limit for the day, Go to https://dashboard.api-football.com to upgrade your plan.'}

🔵 lineups

🔵 events

🔵 statistics

🔵 players

🔵 standings
❌ Erro API: {'requests': 'You have reached the request limit for the day, Go to https://dashboard.api-football.com to upgrade your plan.'}
❌ Erro API: {'requests': 'You have reached the request limit for the day, Go to https://dashboard.api-football.com to upgrade your plan.'}
❌ Erro API: {'requests': 'You have reached the request limit for the day, Go to https://dashboard.api-football.com to upgrade your plan.'}

🔵 team_statistics
❌ Erro API: {'requests': 'You have reached the request limit for the day, Go